In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import scienceplots
import glob

from sklearn.metrics import mean_squared_error, mean_absolute_percentage_error

### MAV

In [2]:
mav_glob = glob.glob('../../outputs/files/models/MAV/*.csv')

In [3]:
ground_truth_file_MAV = pd.read_csv('../../outputs/files/census_pop/MAV_GT_manuscript.csv', sep=";")
ground_truth_file_MAV.dateStart = pd.to_datetime(ground_truth_file_MAV.dateStart)
ground_truth_file_MAV.rename(columns={'final_population':'Nt'}, inplace=True)
raw_data = ground_truth_file_MAV.copy()

In [4]:
# lockdowns only
fl_sd, fl_ed = pd.to_datetime('2020-03-17'), pd.to_datetime('2020-05-10')
sl_sd, sl_ed = pd.to_datetime('2020-10-30'), pd.to_datetime('2020-12-14')

In [5]:
def compute_CR(target, CIL, CIU):
    considered = 0
    inside = 0
    
    for index in range(target.shape[0]):
        considered+=1
        if target[index] >= CIL[index] and target[index] <= CIU[index]:
            inside+=1

    return 100 * inside / considered

def compute_RMSE(target, estimate):
    return np.sqrt(mean_squared_error(target, estimate))

def compute_log_RMSE(target, estimate):
    return np.sqrt(mean_squared_error(np.log(target), np.log(estimate)))

def compute_MAPE(target, estimate):
    return mean_absolute_percentage_error(target, estimate)

def get_mean_of_errors(input_dict):
    output_dict = {}
    for model in list(input_dict.keys()):
        these_errors_mean = np.mean(input_dict[model])
        output_dict[model] = these_errors_mean

    sorted_output_dict = dict(sorted(output_dict.items(), key=lambda x: x[1]))

    return sorted_output_dict

In [6]:
np.random.seed(48)

nb_subsamplings = 1000

target = raw_data.loc[~raw_data['Nt'].isna()]

targets_list = []
for k in range(nb_subsamplings):
    sub_target = target.sample(frac=1.0, replace=True)
    targets_list.append(sub_target)

In [7]:
RMSES_dict_MAV = {}
log_RMSES_dict_MAV = {}
CR_dict_MAV = {}
MAPE_dict_MAV = {}
combi_targets = {}

for model_index, model_path in enumerate(mav_glob):
    model_name = model_path.split('/')[-1].split('.csv')[0]
    
    print('------')
    print(model_index, model_name)
    print('------')   

    estimate = pd.read_csv(model_path, sep=";")
    estimate.dateStart = pd.to_datetime(estimate.dateStart)

    check_targets_list = []
    RMSES_list = []
    RMSES_log_list = []
    MAPES_list = []
    for k in range(nb_subsamplings):
        sub_target = targets_list[k]
        sub_estimate = estimate.copy()
        sub_estimate.set_index('dateStart', inplace=True)
        sub_estimate = sub_estimate.loc[sub_target.dateStart.tolist()]

        this_RMSE = compute_RMSE(sub_target.Nt.values, sub_estimate.Nt_hat.values)
        RMSES_list.append(this_RMSE)

        this_RMSLE = compute_log_RMSE(sub_target.Nt.values, sub_estimate.Nt_hat.values)  
        RMSES_log_list.append(this_RMSLE)

        this_MAPE = compute_MAPE(sub_target.Nt.values, sub_estimate.Nt_hat.values)
        MAPES_list.append(this_MAPE)
        
    RMSES_dict_MAV[model_name] = RMSES_list
    log_RMSES_dict_MAV[model_name] = RMSES_log_list
    MAPE_dict_MAV[model_name] = MAPES_list

------
0 model_5_4_DCO-NGL-MES
------
------
1 model_6_3_DBO-NGL-PT-MES
------
------
2 model_4_6_DCO-NTK-NGL-MES
------
------
3 model_2_1_NH4-DBO-MES
------
------
4 model_3_1_DCO-PT
------
------
5 model_6_3_NH4-NTK-PT-MES
------
------
6 model_6_2_DCO-DBO-NTK-NGL-PT
------
------
7 model_2_1_NH4-DCO-MES
------
------
8 model_6_5_NGL
------
------
9 model_4_3_NH4-DCO-NTK-NGL
------
------
10 model_4_6_NH4-DCO-DBO-NTK-NGL
------
------
11 model_5_0_NH4-PT
------
------
12 model_6_3_NH4-NGL-MES
------
------
13 model_4_5_DCO-DBO-NTK-NGL-MES
------
------
14 model_4_4_NH4-DCO-NTK-NGL-MES
------
------
15 model_5_5_DBO-NTK-NGL-PT-MES
------
------
16 model_6_1_NH4-NGL
------
------
17 model_4_3_NH4-DBO-NGL-PT
------
------
18 model_6_2_DCO-DBO
------
------
19 model_4_4_NH4-DBO-NTK-NGL-PT
------
------
20 model_3_1_NH4-DCO-DBO
------
------
21 model_4_4_NH4-DCO-DBO-NTK-NGL-PT
------
------
22 model_5_4_DCO-NGL-PT-MES
------
------
23 model_1_1_NH4-NTK-NGL
------
------
24 model_6_6_NH4-

### SEV

In [8]:
sev_glob = glob.glob('../../outputs/files/models/SEV/*.csv')

In [9]:
ground_truth_file_SEV = pd.read_csv('../../outputs/files/census_pop/SEV_GT_manuscript.csv', sep=";")
ground_truth_file_SEV.dateStart = pd.to_datetime(ground_truth_file_SEV.dateStart)
ground_truth_file_SEV.rename(columns={'final_population':'Nt'}, inplace=True)
raw_data = ground_truth_file_SEV.copy()

In [10]:
np.random.seed(48)

nb_subsamplings = 1000

target = raw_data.loc[~raw_data['Nt'].isna()]

targets_list = []
for k in range(nb_subsamplings):
    sub_target = target.sample(frac=1.0, replace=True)
    targets_list.append(sub_target)

In [11]:
RMSES_dict_SEV = {}
log_RMSES_dict_SEV = {}
CR_dict_SEV = {}
MAPE_dict_SEV = {}
combi_targets = {}

for model_index, model_path in enumerate(sev_glob):
    model_name = model_path.split('/')[-1].split('.csv')[0]
    
    print('------')
    print(model_index, model_name)
    print('------')   

    estimate = pd.read_csv(model_path, sep=";")
    estimate.dateStart = pd.to_datetime(estimate.dateStart)

    check_targets_list = []
    RMSES_list = []
    RMSES_log_list = []
    MAPES_list = []
    for k in range(nb_subsamplings):
        sub_target = targets_list[k]
        sub_estimate = estimate.copy()
        sub_estimate.set_index('dateStart', inplace=True)
        sub_estimate = sub_estimate.loc[sub_target.dateStart.tolist()]

        this_RMSE = compute_RMSE(sub_target.Nt.values, sub_estimate.Nt_hat.values)
        RMSES_list.append(this_RMSE)

        this_RMSLE = compute_log_RMSE(sub_target.Nt.values, sub_estimate.Nt_hat.values)  
        RMSES_log_list.append(this_RMSLE)

        this_MAPE = compute_MAPE(sub_target.Nt.values, sub_estimate.Nt_hat.values)
        MAPES_list.append(this_MAPE)
        
    RMSES_dict_SEV[model_name] = RMSES_list
    log_RMSES_dict_SEV[model_name] = RMSES_log_list
    MAPE_dict_SEV[model_name] = MAPES_list

------
0 model_5_4_DCO-NGL-MES
------
------
1 model_6_3_DBO-NGL-PT-MES
------
------
2 model_4_6_DCO-NTK-NGL-MES
------
------
3 model_2_1_NH4-DBO-MES
------
------
4 model_3_1_DCO-PT
------
------
5 model_6_3_NH4-NTK-PT-MES
------
------
6 model_6_2_DCO-DBO-NTK-NGL-PT
------
------
7 model_2_1_NH4-DCO-MES
------
------
8 model_6_5_NGL
------
------
9 model_4_3_NH4-DCO-NTK-NGL
------
------
10 model_4_6_NH4-DCO-DBO-NTK-NGL
------
------
11 model_5_0_NH4-PT
------
------
12 model_6_3_NH4-NGL-MES
------
------
13 model_4_5_DCO-DBO-NTK-NGL-MES
------
------
14 model_4_4_NH4-DCO-NTK-NGL-MES
------
------
15 model_5_5_DBO-NTK-NGL-PT-MES
------
------
16 model_6_1_NH4-NGL
------
------
17 model_4_3_NH4-DBO-NGL-PT
------
------
18 model_6_2_DCO-DBO
------
------
19 model_4_4_NH4-DBO-NTK-NGL-PT
------
------
20 model_3_1_NH4-DCO-DBO
------
------
21 model_4_4_NH4-DCO-DBO-NTK-NGL-PT
------
------
22 model_5_4_DCO-NGL-PT-MES
------
------
23 model_1_1_NH4-NTK-NGL
------
------
24 model_6_6_NH4-

### Combined

In [12]:
sorted_RMSES_dict_SEV = dict(sorted(RMSES_dict_SEV.items(), key=lambda x: x[1]))
sorted_log_RMSES_dict_SEV = dict(sorted(log_RMSES_dict_SEV.items(), key=lambda x: x[1]))
sorted_MAPE_dict_SEV = dict(sorted(MAPE_dict_SEV.items(), key=lambda x: x[1]))

In [13]:
sorted_RMSES_dict_MAV = dict(sorted(RMSES_dict_MAV.items(), key=lambda x: x[1]))
sorted_log_RMSES_dict_MAV = dict(sorted(log_RMSES_dict_MAV.items(), key=lambda x: x[1]))
sorted_MAPE_dict_MAV = dict(sorted(MAPE_dict_MAV.items(), key=lambda x: x[1]))

In [14]:
RMSLE_SEV_mean = get_mean_of_errors(sorted_log_RMSES_dict_SEV)
RMSLE_MAV_mean = get_mean_of_errors(sorted_log_RMSES_dict_MAV)

In [15]:
temp = {}
for key in list(RMSLE_SEV_mean.keys()):
    SEV_error = RMSLE_SEV_mean[key]
    MAV_error = RMSLE_MAV_mean[key]
    avg_error = (SEV_error + MAV_error) / 2

    temp[key] = avg_error

RMSLE_overall_mean = dict(sorted(temp.items(), key=lambda x: x[1]))

In [16]:
RMSLE_overall_mean

{'model_5_4_NH4-DCO-DBO-NTK-NGL': 0.0802375763119483,
 'best_model': 0.0802375763119483,
 'model_vn_modified': 0.08065080632617105,
 'model_5_3_NH4-DCO-DBO-NGL': 0.0806998680415697,
 'model_5_3_NH4-DCO-NTK-NGL-MES': 0.08073798268358531,
 'model_5_3_NH4-DCO-DBO-NGL-PT': 0.08082493549730624,
 'model_5_3_NH4-DCO-DBO-NTK': 0.0809306675858886,
 'model_6_3_NH4-DCO-NTK-NGL': 0.08124622918552067,
 'model_4_3_NH4-DCO-NTK-NGL-PT': 0.0812508976930516,
 'model_4_3_NH4-DCO-NTK-NGL': 0.08125198318979986,
 'model_6_3_NH4-DCO-NTK-NGL-PT': 0.08125635107316162,
 'model_6_3_NH4-DCO-DBO-NGL': 0.08127219195721917,
 'model_4_3_NH4-DCO-DBO-NTK-NGL-PT': 0.08128346526371798,
 'model_4_3_NH4-DCO-DBO-NTK-NGL': 0.08128347162056387,
 'model_6_3_NH4-DCO-DBO-NTK-NGL-PT': 0.08129369264425743,
 'model_6_4_NH4-DCO-DBO-NTK-NGL': 0.08130862979898722,
 'model_6_4_NH4-DCO-NTK-NGL-PT': 0.08133063570784507,
 'model_6_3_NH4-DCO-DBO-NTK-NGL': 0.0813336277213334,
 'model_4_3_DCO-NTK-NGL-PT': 0.08133397107578638,
 'model_4_3_DCO

In [17]:
print(f'RMSE best model ({list(RMSLE_overall_mean.keys())[0]}): {np.round(list(RMSLE_overall_mean.values())[0], 4)}')
print(f"RMSE van Nuijs et al., 2011 original: {np.round(RMSLE_overall_mean['model_vn_original'], 4)}")
print(f"RMSE Zheng et al., 2019 original: {np.round(RMSLE_overall_mean['model_zheng_original'], 4)}")
print(f"RMSE Been et al., 2014 original: {np.round(RMSLE_overall_mean['model_been_original'], 4)}")
print("=============")
print(f"RMSE van Nuijs et al., 2011 modified: {np.round(RMSLE_overall_mean['model_vn_modified'], 4)}")
print(f"RMSE Zheng et al., 2019 modified: {np.round(RMSLE_overall_mean['model_zheng_modified'], 4)}")
print(f"RMSE Been et al., 2014 modified: {np.round(RMSLE_overall_mean['model_been_modified'], 4)}")

RMSE best model (model_5_4_NH4-DCO-DBO-NTK-NGL): 0.0802
RMSE van Nuijs et al., 2011 original: 0.1201
RMSE Zheng et al., 2019 original: 0.1555
RMSE Been et al., 2014 original: 0.1629
RMSE van Nuijs et al., 2011 modified: 0.0807
RMSE Zheng et al., 2019 modified: 0.137
RMSE Been et al., 2014 modified: 0.1473
